첫 번째 페이지 데이터를 요청합니다...
{'response': {'header': {'resultCode': '00', 'resultMsg': '정상'}, 'body': {'items': [{'bidNtceNo': 'R25BK00843544', 'bidNtceOrd': '000', 'reNtceYn': 'N', 'rgstTyNm': '조달청 또는 나라장터 자체 공고건', 'ntceKindNm': '등록공고', 'intrbidYn': 'N', 'bidNtceDt': '2025-05-14 06:29:37', 'refNo': '부산광역시 공고 제2025-1719호', 'bidNtceNm': '「한국의 풍물」 작품 유지보수 및 재설치 용역', 'ntceInsttCd': '6260000', 'ntceInsttNm': '부산광역시', 'dminsttCd': '6260000', 'dminsttNm': '부산광역시', 'bidMethdNm': '전자입찰', 'cntrctCnclsMthdNm': '제한경쟁', 'ntceInsttOfclNm': '김성곤', 'ntceInsttOfclTelNo': '051-888-2234', 'ntceInsttOfclEmailAdrs': '', 'exctvNm': '김성곤', 'bidQlfctRgstDt': '2025-06-04 18:00', 'cmmnSpldmdAgrmntRcptdocMethd': '전자', 'cmmnSpldmdAgrmntClseDt': '2025-06-04 18:00:00', 'cmmnSpldmdCorpRgnLmtYn': '', 'bidBeginDt': '2025-05-30 09:00:00', 'bidClseDt': '2025-06-05 10:00:00', 'opengDt': '2025-06-05 11:00:00', 'ntceSpecDocUrl1': 'https://www.g2b.go.kr/pn/pnp/pnpe/UntyAtchFile/downloadFile.do?bidPbancNo=R25BK00843544&bidPbancOr

✅ 엑셀 저장 완료: nara_bid_result.xlsx


In [18]:
import requests
import openpyxl
import json
import math
import time
from datetime import datetime

def get_all_bid_data():
    """나라장터 입찰공고 전체 데이터를 페이지별로 수집하여 Excel로 저장"""
    
    # 1. API 기본 정보 설정
    base_url = "http://apis.data.go.kr/1230000/ad/BidPublicInfoService"
    operation_name = "getBidPblancListInfoServcPPSSrch"
    url = f"{base_url}/{operation_name}"
    
    # 2. ServiceKey 설정 (실제 발급받은 키로 교체 필요)
    service_key = "qPYIYI7lUydoOKfx4lfTj1Bddm3sLaMp0/sbhMz/b9aWP89T9aZZaBsIzd9yapSE/5fyXGO8Q2mpdzBlEGYj+Q=="  # 여기에 실제 ServiceKey 입력
    
    if not service_key:
        print("❌ ServiceKey를 설정해주세요!")
        return
    
    # 3. 기본 파라미터 설정
    rows_per_page = 100  # 한 페이지당 조회할 데이터 수 (API 제한에 맞게 조정)
    
    base_params = {
        'ServiceKey': service_key,
        'type': 'json',
        'inqryDiv': '1',
        'inqryBgnDt': '202505010000', 
        'inqryEndDt': '202505312359',
        'numOfRows': str(rows_per_page),
        'bidNtceNm' : '세무'
    }
    
    try:
        # 4. 첫 번째 페이지 요청으로 전체 데이터 수 확인
        print("📊 첫 번째 페이지 데이터를 요청하여 전체 건수를 확인합니다...")
        first_params = base_params.copy()
        first_params['pageNo'] = '1'
        
        response = requests.get(url, params=first_params, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        # API 응답 구조 확인
        if 'response' not in data or 'body' not in data['response']:
            print("❌ API 응답 구조가 예상과 다릅니다.")
            print("응답 데이터:", json.dumps(data, indent=2, ensure_ascii=False))
            return
        
        body = data['response']['body']
        total_count = int(body.get('totalCount', 0))
        
        if total_count == 0:
            print("❌ 조회된 데이터가 없습니다.")
            return
        
        # 전체 페이지 수 계산
        total_pages = math.ceil(total_count / rows_per_page)
        
        print(f"📈 전체 데이터 건수: {total_count:,}건")
        print(f"📄 전체 페이지 수: {total_pages}페이지")
        print(f"⏱️  예상 소요시간: 약 {total_pages * 2}초 (페이지당 2초 대기)")
        
        # 5. Excel 워크북 생성
        wb = openpyxl.Workbook()
        ws = wb.active
        ws.title = "입찰공고"
        
        # 6. 전체 데이터 수집
        all_items = []
        headers_set = False
        
        for page in range(1, total_pages + 1):
            print(f"🔄 {page}/{total_pages} 페이지 데이터 요청 중...")
            
            # 페이지별 요청 파라미터 설정
            page_params = base_params.copy()
            page_params['pageNo'] = str(page)
            
            try:
                # API 요청
                response = requests.get(url, params=page_params, timeout=30)
                response.raise_for_status()
                page_data = response.json()
                
                # 데이터 추출
                if ('response' in page_data and 
                    'body' in page_data['response'] and 
                    'items' in page_data['response']['body']):
                    
                    items = page_data['response']['body']['items']
                    
                    # 첫 번째 페이지에서 헤더 설정
                    if not headers_set and items:
                        headers = list(items[0].keys())
                        ws.append(headers)
                        headers_set = True
                        print(f"📋 컬럼 헤더 설정 완료: {len(headers)}개 필드")
                    
                    # 데이터 추가
                    for item in items:
                        if headers_set:
                            row = [item.get(h, "") for h in headers]
                            ws.append(row)
                            all_items.append(item)
                    
                    print(f"✅ {page}페이지 완료 - {len(items)}건 수집")
                
                else:
                    print(f"⚠️  {page}페이지에서 데이터를 찾을 수 없습니다.")
                
            except requests.exceptions.RequestException as e:
                print(f"❌ {page}페이지 요청 실패: {e}")
                continue
            except Exception as e:
                print(f"❌ {page}페이지 처리 중 오류: {e}")
                continue
            
            # API 요청 간격 조절 (서버 부하 방지)
            if page < total_pages:
                time.sleep(1)  # 1초 대기
        
        # 7. Excel 파일 저장
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"nara_bid_result_{timestamp}.xlsx"
        
        wb.save(filename)
        
        print(f"\n🎉 데이터 수집 완료!")
        print(f"📁 저장된 파일: {filename}")
        print(f"📊 총 수집 건수: {len(all_items):,}건")
        print(f"📋 컬럼 수: {len(headers) if headers_set else 0}개")
        
        # 수집된 데이터 샘플 출력
        if all_items:
            print(f"\n📝 수집된 데이터 샘플 (첫 번째 항목):")
            for key, value in list(all_items[0].items())[:5]:  # 첫 5개 필드만 출력
                print(f"   {key}: {value}")
            if len(all_items[0]) > 5:
                print(f"   ... 외 {len(all_items[0])-5}개 필드")
    
    except requests.exceptions.RequestException as e:
        print(f"❌ API 요청 오류: {e}")
    except Exception as e:
        print(f"❌ 처리 중 오류 발생: {e}")
        import traceback
        traceback.print_exc()

def main():
    """메인 실행 함수"""
    print("🚀 나라장터 입찰공고 전체 데이터 수집을 시작합니다...")
    print("=" * 60)
    
    # 시작 시간 기록
    start_time = datetime.now()
    
    # 데이터 수집 실행
    get_all_bid_data()
    
    # 종료 시간 계산
    end_time = datetime.now()
    duration = end_time - start_time
    
    print("=" * 60)
    print(f"⏱️  총 실행시간: {duration}")
    print("🏁 프로그램이 완료되었습니다.")

if __name__ == "__main__":
    main()

🚀 나라장터 입찰공고 전체 데이터 수집을 시작합니다...
📊 첫 번째 페이지 데이터를 요청하여 전체 건수를 확인합니다...
📈 전체 데이터 건수: 11건
📄 전체 페이지 수: 1페이지
⏱️  예상 소요시간: 약 2초 (페이지당 2초 대기)
🔄 1/1 페이지 데이터 요청 중...
📋 컬럼 헤더 설정 완료: 109개 필드
✅ 1페이지 완료 - 11건 수집

🎉 데이터 수집 완료!
📁 저장된 파일: nara_bid_result_20250922_164222.xlsx
📊 총 수집 건수: 11건
📋 컬럼 수: 109개

📝 수집된 데이터 샘플 (첫 번째 항목):
   bidNtceNo: R25BK00830430
   bidNtceOrd: 000
   reNtceYn: N
   rgstTyNm: 조달청 또는 나라장터 자체 공고건
   ntceKindNm: 등록공고
   ... 외 104개 필드
⏱️  총 실행시간: 0:00:03.340341
🏁 프로그램이 완료되었습니다.


In [20]:
import requests
import openpyxl
import json
import math
import time
from datetime import datetime

#민간누리장터

def get_all_bid_data():
    """나라장터 입찰공고 전체 데이터를 페이지별로 수집하여 Excel로 저장"""
    
    # 1. API 기본 정보 설정
    base_url = "http://apis.data.go.kr/1230000/ao/PrvtBidNtceService"
    operation_name = "getPrvtBidPblancListInfoServcPPSSrch"
    url = f"{base_url}/{operation_name}"
    
    # 2. ServiceKey 설정 (실제 발급받은 키로 교체 필요)
    service_key = "qPYIYI7lUydoOKfx4lfTj1Bddm3sLaMp0/sbhMz/b9aWP89T9aZZaBsIzd9yapSE/5fyXGO8Q2mpdzBlEGYj+Q=="  # 여기에 실제 ServiceKey 입력
    
    if not service_key:
        print("❌ ServiceKey를 설정해주세요!")
        return
    
    # 3. 기본 파라미터 설정
    rows_per_page = 100  # 한 페이지당 조회할 데이터 수 (API 제한에 맞게 조정)
    
    base_params = {
        'ServiceKey': service_key,
        'type': 'json',
        'inqryDiv': '1',
        'inqryBgnDt': '202505010000', 
        'inqryEndDt': '202505312359',
        'numOfRows': str(rows_per_page),
        'bidNtceNm' : '세무'
    }
    
    try:
        # 4. 첫 번째 페이지 요청으로 전체 데이터 수 확인
        print("📊 첫 번째 페이지 데이터를 요청하여 전체 건수를 확인합니다...")
        first_params = base_params.copy()
        first_params['pageNo'] = '1'
        
        response = requests.get(url, params=first_params, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        # API 응답 구조 확인
        if 'response' not in data or 'body' not in data['response']:
            print("❌ API 응답 구조가 예상과 다릅니다.")
            print("응답 데이터:", json.dumps(data, indent=2, ensure_ascii=False))
            return
        
        body = data['response']['body']
        total_count = int(body.get('totalCount', 0))
        
        if total_count == 0:
            print("❌ 조회된 데이터가 없습니다.")
            return
        
        # 전체 페이지 수 계산
        total_pages = math.ceil(total_count / rows_per_page)
        
        print(f"📈 전체 데이터 건수: {total_count:,}건")
        print(f"📄 전체 페이지 수: {total_pages}페이지")
        print(f"⏱️  예상 소요시간: 약 {total_pages * 2}초 (페이지당 2초 대기)")
        
        # 5. Excel 워크북 생성
        wb = openpyxl.Workbook()
        ws = wb.active
        ws.title = "입찰공고"
        
        # 6. 전체 데이터 수집
        all_items = []
        headers_set = False
        
        for page in range(1, total_pages + 1):
            print(f"🔄 {page}/{total_pages} 페이지 데이터 요청 중...")
            
            # 페이지별 요청 파라미터 설정
            page_params = base_params.copy()
            page_params['pageNo'] = str(page)
            
            try:
                # API 요청
                response = requests.get(url, params=page_params, timeout=30)
                response.raise_for_status()
                page_data = response.json()
                
                # 데이터 추출
                if ('response' in page_data and 
                    'body' in page_data['response'] and 
                    'items' in page_data['response']['body']):
                    
                    items = page_data['response']['body']['items']
                    
                    # 첫 번째 페이지에서 헤더 설정
                    if not headers_set and items:
                        headers = list(items[0].keys())
                        ws.append(headers)
                        headers_set = True
                        print(f"📋 컬럼 헤더 설정 완료: {len(headers)}개 필드")
                    
                    # 데이터 추가
                    for item in items:
                        if headers_set:
                            row = [item.get(h, "") for h in headers]
                            ws.append(row)
                            all_items.append(item)
                    
                    print(f"✅ {page}페이지 완료 - {len(items)}건 수집")
                
                else:
                    print(f"⚠️  {page}페이지에서 데이터를 찾을 수 없습니다.")
                
            except requests.exceptions.RequestException as e:
                print(f"❌ {page}페이지 요청 실패: {e}")
                continue
            except Exception as e:
                print(f"❌ {page}페이지 처리 중 오류: {e}")
                continue
            
            # API 요청 간격 조절 (서버 부하 방지)
            if page < total_pages:
                time.sleep(1)  # 1초 대기
        
        # 7. Excel 파일 저장
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"nara_bid_result_{timestamp}.xlsx"
        
        wb.save(filename)
        
        print(f"\n🎉 데이터 수집 완료!")
        print(f"📁 저장된 파일: {filename}")
        print(f"📊 총 수집 건수: {len(all_items):,}건")
        print(f"📋 컬럼 수: {len(headers) if headers_set else 0}개")
        
        # 수집된 데이터 샘플 출력
        if all_items:
            print(f"\n📝 수집된 데이터 샘플 (첫 번째 항목):")
            for key, value in list(all_items[0].items())[:5]:  # 첫 5개 필드만 출력
                print(f"   {key}: {value}")
            if len(all_items[0]) > 5:
                print(f"   ... 외 {len(all_items[0])-5}개 필드")
    
    except requests.exceptions.RequestException as e:
        print(f"❌ API 요청 오류: {e}")
    except Exception as e:
        print(f"❌ 처리 중 오류 발생: {e}")
        import traceback
        traceback.print_exc()

def main():
    """메인 실행 함수"""
    print("🚀 나라장터 입찰공고 전체 데이터 수집을 시작합니다...")
    print("=" * 60)
    
    # 시작 시간 기록
    start_time = datetime.now()
    
    # 데이터 수집 실행
    get_all_bid_data()
    
    # 종료 시간 계산
    end_time = datetime.now()
    duration = end_time - start_time
    
    print("=" * 60)
    print(f"⏱️  총 실행시간: {duration}")
    print("🏁 프로그램이 완료되었습니다.")

if __name__ == "__main__":
    main()

🚀 나라장터 입찰공고 전체 데이터 수집을 시작합니다...
📊 첫 번째 페이지 데이터를 요청하여 전체 건수를 확인합니다...
📈 전체 데이터 건수: 16건
📄 전체 페이지 수: 1페이지
⏱️  예상 소요시간: 약 2초 (페이지당 2초 대기)
🔄 1/1 페이지 데이터 요청 중...
📋 컬럼 헤더 설정 완료: 71개 필드
✅ 1페이지 완료 - 16건 수집

🎉 데이터 수집 완료!
📁 저장된 파일: nara_bid_result_20250922_164452.xlsx
📊 총 수집 건수: 16건
📋 컬럼 수: 71개

📝 수집된 데이터 샘플 (첫 번째 항목):
   bidNtceNo: R25BK00829586
   bidNtceOrd: 000
   bidNtceClsfc: 민간일반용역
   nticeDt: 2025-05-07 13:45:29
   refNo: 798-22 공고 제2025-14호
   ... 외 66개 필드
⏱️  총 실행시간: 0:00:01.189100
🏁 프로그램이 완료되었습니다.


In [26]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
import time

# 1. 브라우저 열기
driver = webdriver.Chrome()  # 크롬 드라이버 경로 필요시 지정
driver.maximize_window()
driver.get("https://www.g2b.go.kr/")

wait = WebDriverWait(driver, 15)  # 대기 시간 늘림

# 2. 메뉴가 있는 iframe으로 전환
# 브라우저에서 F12 → Elements 확인 후 iframe id 또는 selector 수정 필요
iframe = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "iframe#iframe")) )
driver.switch_to.frame(iframe)

# 3. 상위 메뉴 "입찰" hover
top_menu = wait.until(EC.presence_of_element_located(
    (By.ID, "mf_wfm_gnb_wfm_gnbMenu_genDepth1_1_btn_menuLvl1_span")
))
ActionChains(driver).move_to_element(top_menu).perform()
time.sleep(0.5)  # 메뉴 애니메이션 대기

# 4. 하위 메뉴 "입찰공고" hover 및 클릭
sub_menu = wait.until(EC.visibility_of_element_located(
    (By.ID, "mf_wfm_gnb_wfm_gnbMenu_genDepth1_1_genDepth2_0_btn_menuLvl2_span")
))
ActionChains(driver).move_to_element(sub_menu).click().perform()
time.sleep(0.5)

# 5. 최종 메뉴 "입찰공고목록" 클릭
final_menu = wait.until(EC.element_to_be_clickable(
    (By.ID, "mf_wfm_gnb_wfm_gnbMenu_genDepth1_1_genDepth2_0_genDepth3_0_btn_menuLvl3_span")
))
final_menu.click()

# 6. 게시일자 입력
start_date_input = wait.until(EC.presence_of_element_located((By.ID, "wq_uuid_31076_ibxStrDay")))
start_date_input.clear()
start_date_input.send_keys("20250514")  # 시작 날짜

end_date_input = driver.find_element(By.ID, "wq_uuid_31076_ibxEndDay")
end_date_input.clear()
end_date_input.send_keys("20250514")  # 종료 날짜

# 7. 공고명 검색어 입력
search_input = driver.find_element(By.ID, "mf_wfm_container_tacBidPbancLst_contents_tab2_body_bidPbancNm")
search_input.clear()
search_input.send_keys("이전가격")

# 8. 검색 버튼 클릭
search_button = driver.find_element(By.ID, "mf_wfm_container_tacBidPbancLst_contents_tab2_body_btnS0004")
search_button.click()

# 9. 결과 화면 20초간 대기
time.sleep(20)

# 10. iframe 밖으로 나가기
driver.switch_to.default_content()

# 11. 종료
driver.quit()


TimeoutException: Message: 
Stacktrace:
	GetHandleVerifier [0x0x7ff662a530f5+79493]
	GetHandleVerifier [0x0x7ff662a53150+79584]
	(No symbol) [0x0x7ff6627d01ba]
	(No symbol) [0x0x7ff662828067]
	(No symbol) [0x0x7ff66282832c]
	(No symbol) [0x0x7ff66287be27]
	(No symbol) [0x0x7ff66285074f]
	(No symbol) [0x0x7ff662878b8b]
	(No symbol) [0x0x7ff6628504e3]
	(No symbol) [0x0x7ff662818e92]
	(No symbol) [0x0x7ff662819c63]
	GetHandleVerifier [0x0x7ff662d10dbd+2954061]
	GetHandleVerifier [0x0x7ff662d0b02a+2930106]
	GetHandleVerifier [0x0x7ff662d2b357+3061991]
	GetHandleVerifier [0x0x7ff662a6d60e+187294]
	GetHandleVerifier [0x0x7ff662a7557f+219919]
	GetHandleVerifier [0x0x7ff662a5c294+116772]
	GetHandleVerifier [0x0x7ff662a5c449+117209]
	GetHandleVerifier [0x0x7ff662a42618+11176]
	BaseThreadInitThunk [0x0x7ffb2923259d+29]
	RtlUserThreadStart [0x0x7ffb2b20af78+40]


In [ ]:
import requests

# 1. 세션 생성
session = requests.Session()

# 2. 요청 헤더 (브라우저에서 확인 후 실제 값으로 수정)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/140.0.0.0 Safari/537.36",
    "Content-Type": "application/json; charset=UTF-8",
    # 필요시 Referer/Cookie 추가
    # "Referer": "https://www.g2b.go.kr/...",
    # "Cookie": "SESSION=xxxx; ..."
}

# 3. 엑셀 다운로드용 URL (개발자 도구 Network 확인 필요)
url = "https://www.g2b.go.kr/pn/pnp/pnpe/BidPbac/selectBidPbacListExcel.do"  # 예시 URL, 실제 확인 필요

# 4. 페이로드 설정
payload = {
    "dlBidPbancLstM": {
        "bidDateType": "R",
        "bsneAllYn": "Y",
        "cangParmVal": "untySrch001",
        "contxtSeCd": "콘010006",
        "currentPage": "",
        "fromBidDt": "20250514",
        "toBidDt": "20250514",
        "frcpYn": "Y",
        "laseYn": "Y",
        "pbancKndCd": "공440002",
        "prcmBsneSeCd": "0000 조070001 조070002 조070003 조070004 조070005 민079999",
        "recordCountPerPage": "10",
        "rsrvYn": "Y",
        "srchTy": "0",
        "startIndex": 1,
        "endIndex": 10,
        # 나머지 필드들은 비워두거나 필요 시 추가
    }
}

# 5. POST 요청
response = session.post(url, headers=headers, json=payload)

# 6. 응답 저장 (엑셀 파일로)
if response.status_code == 200:
    with open("입찰공고.xlsx", "wb") as f:
        f.write(response.content)
    print("엑셀 다운로드 완료: 입찰공고.xlsx")
else:
    print(f"요청 실패: {response.status_code}")


엑셀 다운로드 완료: 입찰공고.xlsx


In [9]:
import requests
import json

# --------------------------
# 1. 세션 생성 및 쿠키 적용
# --------------------------
session = requests.Session()

cookie_str = "WHATAP=x3mkb31uiipeb9; XTVID=A2509190941317491; _harry_fid=hh-208357241; _harry_lang=en-US; JSESSIONID=ZmE3ZTAxNjktOTUzOS00MTE2LWE2NGMtOTcxNWY3NjVlYWZm; infoSysCd=%EC%A0%95010029; xloc=1920X1080; system_language=en; poupR23AB00000134353=done; poupR23AB00000134350=done; poupR23AB00000134343=done; poupR23AB00000134324=done; poupR23AB00000134241=done; lastAccess=1758520150506; globalDebug=false; XTSID=A250922172400178768; _harry_ref=https://www.google.com/; _harry_url=https://www.g2b.go.kr/; _harry_hsid=A250922175928503379; _harry_dsid=A250922175928504373"

cookies = {item.split("=")[0]: item.split("=")[1] for item in cookie_str.split("; ")}
session.cookies.update(cookies)

# --------------------------
# 2. 엑셀 다운로드 요청
# --------------------------
excel_url = "https://www.g2b.go.kr/pn/pnp/pnpe/BidPbac/selectBidPbacListExcel.do"
payload = {
    "dlBidPbancLstM": {
        "bidDateType": "R",
        "bidPbancNm": "이전가격",
        "bsneAllYn": "Y",
        "frcpYn": "Y",
        "fromBidDt": "20250514",
        "toBidDt": "20250514",
        "infoSysCd": "정010029",
        "laseYn": "Y",
        "pbancKndCd": "공440002",
        "prcmBsneSeCd": "0000 조070001 조070002 조070003 조070004 조070005 민079999",
        "recordCountPerPage": "10",
        "rsrvYn": "Y",
        "srchTy": "0",
        "startIndex": 1,
        "endIndex": 10
    }
}

headers = {
    "Content-Type": "application/json;charset=UTF-8",
    "Origin": "https://www.g2b.go.kr",
    "Referer": "https://www.g2b.go.kr/"
}

response = session.post(excel_url, headers=headers, data=json.dumps(payload))

print("엑셀 다운로드 요청 상태:", response.status_code)

# --------------------------
# 3. 응답이 200이면 파일로 저장
# --------------------------
if response.status_code == 200:
    with open("bid_list.xlsx", "wb") as f:
        f.write(response.content)
    print("엑셀 파일 저장 완료: bid_list.xlsx")
else:
    print("엑셀 다운로드 실패:", response.text)


엑셀 다운로드 요청 상태: 500
엑셀 다운로드 실패: {"ErrorMsg":"조회하는데 에러가 발생하였습니다.","reason":"Internal Server Error","path":"/pn/pnp/pnpe/BidPbac/selectBidPbacListExcel.do","ErrorCode":-1,"locale":"ko"}
